# ESCI Embedding Generation

Generates dense embeddings for the ESCI product catalogue using `sentence-transformers` on Colab GPU.

**Runtime**: Runtime → Change runtime type → **T4 GPU**

## Where the files are stored

```
Google Colab GPU (RAM)
        ↓  model.encode(...)
Google Drive  →  MyDrive/antigravity-esci/vectorstore/embeddings/
                    product_embeddings.npy   ← float32 matrix (N_products × dim)
                    product_ids.npy          ← string array, same row order as embeddings
                    query_embeddings.npy     ← float32 matrix (N_queries × dim)
                    query_ids.npy            ← string array, same row order as embeddings
                    embedding_meta.json      ← model name, dim, counts
        ↓  download (cell 14) or copy from Drive
Local machine  →  antigravity-esci/vectorstore/embeddings/
```

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers pyarrow pandas tqdm

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

Edit these paths / settings before running the rest of the notebook.

In [ ]:
import os

# ── Data paths ──────────────────────────────────────────────────────────────
DRIVE_DATA_DIR = '/content/drive/MyDrive/antigravity-esci/data/raw'
# DRIVE_DATA_DIR = '/content'   # ← uncomment if files uploaded directly to Colab

EXAMPLES_PATH = os.path.join(DRIVE_DATA_DIR, 'shopping_queries_dataset_examples.parquet')
PRODUCTS_PATH = os.path.join(DRIVE_DATA_DIR, 'shopping_queries_dataset_products.parquet')

# ── Output dir ────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/antigravity-esci/vectorstore/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Embedding model ───────────────────────────────────────────────────────────
MODEL_NAME = 'all-MiniLM-L6-v2'

# ── Locale filter ─────────────────────────────────────────────────────────────
LOCALE = 'us'

# ── Batch size — 1024 fits comfortably in T4 16 GB VRAM ──────────────────────
BATCH_SIZE = 1024

# ── fp16 — ~2x faster on GPU, negligible quality loss ────────────────────────
USE_FP16 = True

# ── Sample mode ───────────────────────────────────────────────────────────────
SAMPLE_N_PRODUCTS = None  # set e.g. 10_000 for a quick test run

print('Config OK')
print(f'  model      : {MODEL_NAME}')
print(f'  locale     : {LOCALE}')
print(f'  batch_size : {BATCH_SIZE}')
print(f'  fp16       : {USE_FP16}')
print(f'  output     : {OUTPUT_DIR}')

## 4. (Optional) Upload data files directly to Colab

Skip this cell if you already copied the files to your Drive at `DRIVE_DATA_DIR`.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # select both .parquet files
# EXAMPLES_PATH = '/content/shopping_queries_dataset_examples.parquet'
# PRODUCTS_PATH = '/content/shopping_queries_dataset_products.parquet'

## 6. Load ESCI data

In [ ]:
import pandas as pd

print('Loading examples...')
examples_df = pd.read_parquet(EXAMPLES_PATH)
print(f'  examples shape : {examples_df.shape}')
print(f'  columns        : {examples_df.columns.tolist()}')

print('\nLoading products...')
products_df = pd.read_parquet(PRODUCTS_PATH)
print(f'  products shape : {products_df.shape}')
print(f'  columns        : {products_df.columns.tolist()}')

## 7. Filter & preprocess

In [ ]:
import re

def clean_text(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'<[^>]+>', ' ', str(text))  # strip HTML
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_product_text(row):
    parts = [
        clean_text(row.get('product_title', '')),
        clean_text(row.get('product_description', '')),
        clean_text(row.get('product_bullet_point', '')),
    ]
    return ' '.join(p for p in parts if p)[:1024]  # 1024 chars ~ 256 tokens (model max)

# ── Filter to chosen locale ────────────────────────────────────────────────
products_locale = products_df[products_df['product_locale'] == LOCALE].copy()
examples_locale = examples_df[examples_df['product_locale'] == LOCALE].copy()
print(f'Products in locale "{LOCALE}": {len(products_locale):,}')
print(f'Examples in locale "{LOCALE}": {len(examples_locale):,}')

# ── Keep only products that appear in examples ─────────────────────────────
relevant_pids = set(examples_locale['product_id'].unique())
products_filtered = products_locale[products_locale['product_id'].isin(relevant_pids)].copy()
products_filtered = products_filtered.drop_duplicates(subset=['product_id'])
print(f'Unique products referenced in examples: {len(products_filtered):,}')

# ── Optional sample ────────────────────────────────────────────────────────
if SAMPLE_N_PRODUCTS is not None:
    products_filtered = products_filtered.sample(n=min(SAMPLE_N_PRODUCTS, len(products_filtered)), random_state=42)
    print(f'Sampled {len(products_filtered):,} products for quick test')

# ── Build text column ──────────────────────────────────────────────────────
print('\nBuilding product text...')
products_filtered['product_text'] = products_filtered.apply(build_product_text, axis=1)
products_filtered = products_filtered[products_filtered['product_text'].str.len() > 0]

# Reset index so iloc[i] matches embedding row i exactly
products_filtered = products_filtered.reset_index(drop=True)

print(f'Products with non-empty text: {len(products_filtered):,}')
print('\nSample product text:')
print(products_filtered['product_text'].iloc[0])

## 8. Prepare unique queries

In [ ]:
queries_df = examples_locale[['query_id', 'query']].drop_duplicates(subset=['query_id']).copy()
queries_df = queries_df.dropna(subset=['query'])
queries_df['query'] = queries_df['query'].apply(clean_text)
queries_df = queries_df[queries_df['query'].str.len() > 0]
print(f'Unique queries: {len(queries_df):,}')

## 9. Load embedding model

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found! Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

device = 'cuda'
model = SentenceTransformer(MODEL_NAME, device=device)   # explicitly pinned to GPU

print(f'Model   : {MODEL_NAME}')
print(f'Device  : {next(model.parameters()).device}')    # confirms cuda:0
print(f'Emb dim : {model.get_sentence_embedding_dimension()}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 10. Generate product embeddings

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found! Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

model = SentenceTransformer(MODEL_NAME, device='cuda')

if USE_FP16:
    model.half()   # convert weights to float16 — ~2x faster on T4

print(f'Model   : {MODEL_NAME}')
print(f'Device  : {next(model.parameters()).device}')
print(f'dtype   : {next(model.parameters()).dtype}')   # float16 if USE_FP16
print(f'Emb dim : {model.get_sentence_embedding_dimension()}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 11. Generate query embeddings

In [ ]:
import numpy as np
import time

product_texts = products_filtered['product_text'].tolist()
product_ids   = products_filtered['product_id'].tolist()

print(f'Embedding {len(product_texts):,} products  (batch={BATCH_SIZE}, fp16={USE_FP16})...')
t0 = time.time()

product_embeddings = model.encode(
    product_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

elapsed = time.time() - t0
print(f'Done in {elapsed/60:.1f} min  |  shape: {product_embeddings.shape}  |  dtype: {product_embeddings.dtype}')

## 12. Quick sanity check (numpy dot product)

In [ ]:
query_texts = queries_df['query'].tolist()
query_ids   = queries_df['query_id'].tolist()

print(f'Embedding {len(query_texts):,} queries  (batch={BATCH_SIZE}, fp16={USE_FP16})...')
t0 = time.time()

query_embeddings = model.encode(
    query_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f} sec  |  shape: {query_embeddings.shape}  |  dtype: {query_embeddings.dtype}')

## 13. Save to Drive

In [ ]:
import json

np.save(os.path.join(OUTPUT_DIR, 'product_embeddings.npy'), product_embeddings)
np.save(os.path.join(OUTPUT_DIR, 'product_ids.npy'),        np.array(product_ids))

np.save(os.path.join(OUTPUT_DIR, 'query_embeddings.npy'),   query_embeddings)
np.save(os.path.join(OUTPUT_DIR, 'query_ids.npy'),          np.array(query_ids))

meta = {
    'model_name'   : MODEL_NAME,
    'locale'       : LOCALE,
    'embedding_dim': int(product_embeddings.shape[1]),
    'n_products'   : int(product_embeddings.shape[0]),
    'n_queries'    : int(query_embeddings.shape[0]),
    'normalized'   : True,
}
with open(os.path.join(OUTPUT_DIR, 'embedding_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to', OUTPUT_DIR)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:40s}  {size_mb:7.1f} MB')

## 14. (Optional) Download to local machine

Use this cell if you want to download the files directly instead of pulling from Drive.

In [ ]:
# from google.colab import files
# for fname in ['product_embeddings.npy', 'product_ids.npy',
#               'query_embeddings.npy',   'query_ids.npy',
#               'embedding_meta.json']:
#     files.download(os.path.join(OUTPUT_DIR, fname))